# Pancancer enrichment analysis step 7: Make figures from GSEApy data with weighted score type = 0

## Setup

In [1]:
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd
import operator
import IPython.display

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Step 1 output
STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

# Step 5 outputs
STEP5_DIR = "step5_outputs"

STEP5_abs_signal_to_noise = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_abs_signal_to_noise.tsv")
STEP5_diff_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_diff_of_classes.tsv")
STEP5_log2_ratio_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_log2_ratio_of_classes.tsv")
STEP5_ratio_of_classes = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_ratio_of_classes.tsv")
STEP5_signal_to_noise = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_signal_to_noise.tsv")
STEP5_t_test = os.path.join(STEP5_DIR, "enrichment_gseapy_reactome_20200618_104458_10000_perms_wst0_t_test.tsv")

# Step 6 output
STEP6_DIR = "step6_outputs"

STEP6_S2N = "enrichment_gseapy_kegg_signal_to_noise_10000_perms_20200617_072701.tsv"
STEP6_S2N_PATH = os.path.join(STEP6_DIR, STEP6_S2N)

STEP6_ABS_S2N = "enrichment_gseapy_kegg_abs_signal_to_noise_10000_perms_20200617_103905.tsv"
STEP6_ABS_S2N_PATH = os.path.join(STEP6_DIR, STEP6_ABS_S2N)

MSV = 0.5 # Simplified expression value for marginally significant proteins

In [3]:
# Altair options
# alt.data_transformers.disable_max_rows()
# alt.data_transformers.enable('json')

## Function to plot top pathways across cancer types

In [4]:
def plot_top_ten(enrich_file_path, expr_file_path, xtitle, fdr=0.3):
    
    # Read in the expression data
    all_expression_data = pd.read_csv(expr_file_path, sep="\t", index_col=0)

    # Make a column where all increases are +1 and all decreases 
    # are -1, because these are ratioed abundances, so we can't 
    # compare magnitudes between genes--we can only compare whether 
    # there was a change, and whether it was positive or negative
    all_expression_data = all_expression_data.assign(simplified_change=np.nan)

    # adj p < 0.05 and change > 1 => +1
    all_expression_data.loc[
        (all_expression_data["change"] > 0) & (all_expression_data["adj_p"] < 0.05), 
        "simplified_change"
    ] = 1

    # adj p >= 0.05 and change > 1 => +0.5
    all_expression_data.loc[(all_expression_data["change"] > 0) & (all_expression_data["adj_p"] >= 0.05),
        "simplified_change"
    ] = MSV

    # change == 0 => 0
    all_expression_data.loc[
        all_expression_data["change"] == 0,
        "simplified_change"
    ] = 0

    # adj p >= 0.05 and change < 1 => -0.5
    all_expression_data.loc[(all_expression_data["change"] < 0) & (all_expression_data["adj_p"] >= 0.05), 
        "simplified_change"
    ] = -MSV

    # adj p < 0.05 and change < 1 => -1
    all_expression_data.loc[
        (all_expression_data["change"] < 0) & (all_expression_data["adj_p"] < 0.05),
        "simplified_change"
    ] = -1

    # Select just the proteins where we chose to reject the null hypothesis of no change
    expression_data = all_expression_data[all_expression_data["reject_null"]]

    # Read in the enrichment data
    enrichment_data = pd.read_csv(enrich_file_path, sep="\t")
    
    # Select enrichment data below our FDR
    enrichment_data = enrichment_data.loc[enrichment_data["fdr"] <= fdr, :]
    
    # Make a column for abs_nes
    enrichment_data = enrichment_data.assign(abs_nes=enrichment_data["nes"].abs())

    # Assign pathway ranks within each cancer type based on absolute value of NES
    enrichment_data = enrichment_data.\
        assign(cancer_rank_abs_nes=enrichment_data.groupby("cancer_type")["abs_nes"].rank(ascending=False)).\
        sort_values(by=["cancer_type", "cancer_rank_abs_nes"]).\
        reset_index(drop=True)

    # Make a table with summary info for all pathways
    enrichment_summary = enrichment_data[["Term"]].drop_duplicates(keep="first")

    pathway_times_enriched = enrichment_summary["Term"].apply(
        lambda x: enrichment_data[enrichment_data["Term"] == x].shape[0])

    avg_rank = enrichment_summary["Term"].apply(
        lambda x: enrichment_data.loc[enrichment_data["Term"] == x, "cancer_rank_abs_nes"].mean())

    enrichment_summary = enrichment_summary.\
        assign(
            pathway_times_enriched=pathway_times_enriched,
            pathway_avg_rank=avg_rank).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank"],
            ascending=[False, True]).\
        reset_index(drop=True)

    # Merge in the original enrichment data
    enrichment_data = enrichment_data.\
        merge(
            enrichment_summary,
            how="outer",
            left_on="Term",
            right_on="Term",
            validate="many_to_one"
        ).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank", "cancer_type"],
            ascending=[False, True, True]
        )

    # Select top 10 for our plot
    top_ten = enrichment_data["Term"].unique()[:10]
    sel_enrich = enrichment_data[enrichment_data["Term"].isin(top_ten)]

    # Calculate the mean expression for each pathway in each cancer type
    mean_exprs = []

    for idx in sel_enrich.index:
        genes = sel_enrich.loc[idx, "genes"].split(";")
        cancer_type = sel_enrich.loc[idx, "cancer_type"]

        genes_expr = expression_data.loc[
            expression_data["protein_str"].isin(genes),
            "simplified_change"
        ].mean()

        mean_exprs.append(genes_expr)

    sel_enrich = sel_enrich.assign(mean_expr=mean_exprs)

    sel_enrich = sel_enrich.assign(
        rank_size=1 / sel_enrich["cancer_rank_abs_nes"],
        avg_rank_size=1 / sel_enrich["pathway_avg_rank"])

    individual = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "Term:N",
            sort=sel_enrich["Term"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="",
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            axis=alt.Axis(
                title="Cancer type",
                titleFontSize=12
            ),
        ),
        size=alt.Size(
            "rank_size:Q",
            legend=None
        ),
        color=alt.Color(
            "mean_expr:Q",
            scale=alt.Scale(
                scheme="blueorange",
                domain=[-1, 1]
            ),
            legend=alt.Legend(
                title="Pathway tumor expression"
            )
        )
    ).properties(
        width=400,
        height=300
    )

    aggregate = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "pathway_avg_rank:N",
            sort=sel_enrich["pathway_avg_rank"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="Overall rank of pathway",
                titleFontSize=12
            )
        ),
        size=alt.Size(
            "avg_rank_size:Q",
            legend=None
        ),
    ).properties(
        width=400
    )

    full_plot = alt.vconcat(
        aggregate, individual
    ).properties(
        title=xtitle
    )
    
    return full_plot, enrichment_data, all_expression_data, enrichment_summary

In [5]:
plot_top_ten(STEP5_abs_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Nervous system development,6,17.833333
1,mRNA 3'-end processing,6,24.500000
2,Hemostasis,6,26.500000
3,Transport of Mature mRNA derived from an Intro...,6,33.833333
4,Cellular response to heat stress,6,40.500000
5,Translation,6,48.000000
6,Regulation of HSF1-mediated heat shock response,6,53.000000
7,Response to elevated platelet cytosolic Ca2+,6,90.333333
8,Binding and Uptake of Ligands by Scavenger Rec...,6,92.500000
9,Influenza Infection,6,93.666667


In [6]:
plot_top_ten(STEP5_diff_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,"Platelet activation, signaling and aggregation",5,93.000
1,NCAM signaling for neurite out-growth,5,93.200
2,G alpha (z) signalling events,5,108.200
3,GPCR downstream signalling,4,12.500
4,Signaling by GPCR,4,16.000
5,Ca2+ pathway,4,35.250
6,Integration of energy metabolism,4,39.750
7,Platelet homeostasis,4,45.500
8,Glucagon signaling in metabolic regulation,4,47.000
9,Activation of GABAB receptors,4,50.500


In [7]:
plot_top_ten(STEP5_log2_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,RAB geranylgeranylation,6,169.666667
1,Viral Messenger RNA Synthesis,5,110.400000
2,RNA Polymerase III Chain Elongation,5,117.200000
3,SUMOylation of ubiquitinylation proteins,5,136.000000
4,RNA Polymerase III Transcription Termination,5,136.800000
5,Rev-mediated nuclear export of HIV RNA,5,144.000000
6,Nuclear Receptor transcription pathway,5,147.200000
7,Interactions of Rev with host cellular proteins,5,152.800000
8,Degradation of AXIN,5,154.400000
9,Regulation of RUNX3 expression and activity,5,162.200000


In [8]:
plot_top_ten(STEP5_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Translation,2,4.0
1,Advanced glycosylation endproduct receptor sig...,2,27.5
2,Activated PKN1 stimulates transcription of AR ...,2,39.5
3,Metabolism of non-coding RNA,2,55.0
4,snRNP Assembly,2,55.0
5,Interactions of Vpr with host cellular proteins,2,62.0
6,Vpr-mediated nuclear import of PICs,2,72.0
7,SUMOylation of RNA binding proteins,2,79.0
8,Transport of the SLBP independent Mature mRNA,2,96.0
9,AMPK inhibits chREBP transcriptional activatio...,2,112.0


In [9]:
plot_top_ten(STEP5_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Major pathway of rRNA processing in the nucleo...,6,34.500000
1,rRNA processing,6,38.833333
2,Influenza Infection,6,64.833333
3,Influenza Viral RNA Transcription and Replication,6,94.833333
4,mRNA 3'-end processing,5,19.600000
5,mRNA Splicing - Major Pathway,5,22.000000
6,mRNA Splicing,5,31.000000
7,HIV Infection,5,40.400000
8,rRNA processing in the nucleus and cytosol,5,45.400000
9,HIV Life Cycle,5,46.000000


In [10]:
plot_top_ten(STEP5_t_test, STEP1_FILE_PATH, "Reactome data - proteins ranked by t_test")[3]

,Term,pathway_times_enriched,pathway_avg_rank
0,Major pathway of rRNA processing in the nucleo...,6,32.000000
1,rRNA processing,6,36.166667
2,Influenza Infection,6,67.833333
3,Influenza Viral RNA Transcription and Replication,6,97.500000
4,G2/M Checkpoints,6,134.000000
5,mRNA 3'-end processing,5,19.800000
6,mRNA Splicing,5,31.600000
7,Transport of the SLBP independent Mature mRNA,5,40.200000
8,rRNA processing in the nucleus and cytosol,5,42.400000
9,HIV Life Cycle,5,45.400000


In [11]:
df = pd.read_csv(STEP5_log2_ratio_of_classes, sep="\t", index_col=0)

In [12]:
df["fdr"].min()

0.0

In [13]:
alt.vconcat(
    plot_top_ten(STEP5_abs_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by abs_signal_to_noise")[0],
    plot_top_ten(STEP5_diff_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by diff_of_classes")[0],
    plot_top_ten(STEP5_log2_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by log2_ratio_of_classes")[0],
    plot_top_ten(STEP5_ratio_of_classes, STEP1_FILE_PATH, "Reactome data - proteins ranked by ratio_of_classes")[0],
    plot_top_ten(STEP5_signal_to_noise, STEP1_FILE_PATH, "Reactome data - proteins ranked by signal_to_noise")[0],
    plot_top_ten(STEP5_t_test, STEP1_FILE_PATH, "Reactome data - proteins ranked by t_test")[0],
    
    plot_top_ten(STEP6_ABS_S2N_PATH, STEP1_FILE_PATH, "KEGG data - proteins ranked by |S2N|")[0],
    plot_top_ten(STEP6_S2N_PATH, STEP1_FILE_PATH, "KEGG data - proteins ranked by S2N")[0],
).configure_axis(
    grid=True
).configure_title(
    fontSize=16,
    anchor="end",
    offset=20
).resolve_scale(
    size="independent"
)

alt.VConcatChart(...)

In [14]:
def find_neutrophil_degranulation(fp):
    enrich = plot_top_ten(fp, STEP1_FILE_PATH, "Reactome data - proteins ranked by |S2N|", fdr=1)[1]

    # enrich = enrich[enrich["pathway_times_enriched"] == 8]
    ranks = enrich[["Term", "pathway_avg_rank"]].\
        set_index("Term").\
        drop_duplicates(keep="first").\
        rank()["pathway_avg_rank"].\
        rename("pathway_overall_rank")

    enrich = enrich.merge(
        right=ranks,
        left_on="Term",
        right_on="Term",
        validate="many_to_one")

    return enrich[enrich["Term"] == "Neutrophil degranulation"]

In [15]:
find_neutrophil_degranulation(STEP5_abs_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
6008,Neutrophil degranulation,0.004765,4.506603,0.021297,0.379019,479,427,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,PYGL;GGH;NDUFC2;TYROBP;FCER1G;ITGAX;RAP2B;LPCA...,ccrcc,4.506603,26.0,8,1070.125,810.0
6009,Neutrophil degranulation,0.001705,1.288740,0.868132,0.932818,479,405,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,METTL7A;S100A11;HSP90AB1;AGL;CNN2;RAB5B;CFD;XR...,colon,1.288740,2044.0,8,1070.125,810.0
6010,Neutrophil degranulation,0.002989,2.680652,0.278468,0.422340,479,427,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,AHSG;EEF2;SERPINA1;TIMP2;COPB1;A1BG;PSMD2;DYNL...,endometrial,2.680652,670.0,8,1070.125,810.0
6011,Neutrophil degranulation,0.002578,1.896457,0.591257,0.756377,479,432,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,AP2A2;HSPA8;TOLLIP;ATP6V1D;LRRC7;NDUFC2;KCNAB2...,gbm,1.896457,1382.0,8,1070.125,810.0
6012,Neutrophil degranulation,0.007681,6.302835,0.000000,0.117560,479,448,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,SPTAN1;AHSG;TTR;CFD;ORM2;A1BG;SERPINA1;CNN2;RA...,hnscc,6.302835,3.0,8,1070.125,810.0
6013,Neutrophil degranulation,-0.001616,-1.097103,0.833989,0.991152,479,423,SPTAN1;EEF2;VAT1;CD93;CD55;CAT;SERPINB6;PECAM1...,HVCN1;MME;MAGT1;ERP44;CREG1;OSCAR;CTSB;TBC1D10...,lscc,1.097103,2168.0,8,1070.125,810.0
6014,Neutrophil degranulation,0.001567,1.201670,0.827846,0.978272,479,409,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,SPTAN1;CAT;PECAM1;VCL;ANXA2;CD36;HBB;CD93;RAP1...,luad,1.201670,2111.0,8,1070.125,810.0
6015,Neutrophil degranulation,0.004403,3.879352,0.060136,0.146381,479,418,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,HSP90AB1;STOM;PRSS3;HSP90AA1;ILF2;COPB1;PECAM1...,ovarian,3.879352,157.0,8,1070.125,810.0


In [16]:
find_neutrophil_degranulation(STEP5_diff_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
4648,Neutrophil degranulation,0.007110,2.916172,0.110717,0.432739,479,427,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,FCER1G;TYROBP;FTL;PYGL;FTH1;ITGAX;SLC2A3;CYBB;...,ccrcc,2.916172,290.0,8,905.75,589.0
4649,Neutrophil degranulation,0.003611,1.361710,0.687558,0.877918,479,405,S100P;S100A9;S100A8;S100A11;S100A12;CAMP;LCN2;...,S100P;S100A9;S100A8;S100A11;S100A12;CAMP;LCN2;...,colon,1.361710,2038.0,8,905.75,589.0
4650,Neutrophil degranulation,0.007459,3.010094,0.150508,0.472200,479,427,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,S100A8;S100A9;FCER1G;LTF;S100A12;DSP;LCN2;CAMP...,endometrial,3.010094,416.0,8,905.75,589.0
4651,Neutrophil degranulation,0.007662,2.595841,0.263342,0.463252,479,432,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,SERPINA1;HBB;S100A11;ANXA2;FCER1G;CD44;SERPINA...,gbm,2.595841,873.0,8,905.75,589.0
4652,Neutrophil degranulation,0.003426,1.396537,0.981333,0.982389,479,448,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,MGAM;AOC1;CEP290;CR1;FCER1G;CAMP;PLAU;EPX;MCEM...,hnscc,1.396537,2032.0,8,905.75,589.0
4653,Neutrophil degranulation,-0.007284,-2.930507,0.227658,0.353054,479,423,DSP;PKP1;PLAU;SERPINB3;CKAP4;JUP;TNFAIP6;S100A...,TNFRSF1B;PFKL;HSPA1A;ADGRG3;HSPA6;DYNLL1;DEFA4...,lscc,2.930507,518.0,8,905.75,589.0
4654,Neutrophil degranulation,-0.006764,-2.964457,0.179889,0.338823,479,409,DSP;PLAU;TXNDC5;CKAP4;MLEC;NEU1;DNAJC3;FUCA1;H...,GOLGA7;TRAPPC1;RAB7A;YPEL5;ACLY;PDXK;IST1;TNFR...,luad,2.964457,425.0,8,905.75,589.0
4655,Neutrophil degranulation,-0.005635,-2.796377,0.210771,0.413294,479,418,NPC2;PKM;HSP90AB1;ILF2;MIF;SRP14;HSP90AA1;CD47...,PLEKHO2;HEBP2;ROCK1;CTSH;ARHGAP9;SDCBP;CR1;CPP...,ovarian,2.796377,654.0,8,905.75,589.0


In [17]:
find_neutrophil_degranulation(STEP5_log2_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
3200,Neutrophil degranulation,0.003483,2.695486,0.202147,0.628476,479,427,ERCC6L;ZNF121;TERF2IP;POLR3D;BAAT;SMG7;IL6ST;C...,ERCC6L;ZNF121;TERF2IP;POLR3D;BAAT;SMG7;IL6ST;C...,ccrcc,2.695486,609.0,8,826.5,412.0
3201,Neutrophil degranulation,-0.005050,-2.572777,0.042137,0.337050,479,405,FBXO22;CHUK;ABAT;HSPA4;LFNG;PLXNA1;GIT1;CDS1;U...,PDXDC1;PDZD11;PFN1;PGAM1;PGM1;PGM3;PGM5;PGP;PH...,colon,2.572777,1052.0,8,826.5,412.0
3202,Neutrophil degranulation,0.002896,10.182593,0.239544,0.054197,479,427,PRPF31;ABCC10;AKR1D1;CPNE9;C9orf40;PPM1J;ALG11...,PRPF31;ABCC10;AKR1D1;CPNE9;C9orf40;PPM1J;ALG11...,endometrial,10.182593,195.0,8,826.5,412.0
3203,Neutrophil degranulation,0.001957,1.328865,0.861328,0.868842,479,432,CASTOR1;ELN;CC2D2A;SHC4;AMACR;C16orf87;PSAT1;T...,CASTOR1;ELN;CC2D2A;SHC4;AMACR;C16orf87;PSAT1;T...,gbm,1.328865,1982.0,8,826.5,412.0
3204,Neutrophil degranulation,0.004906,1.833549,0.602496,0.828161,479,448,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,PTPRN2;P2RX1;AOC1;MMP25;NFASC;MCEMP1;MGAM;CEP2...,hnscc,1.833549,1624.0,8,826.5,412.0
3205,Neutrophil degranulation,-0.002328,-8.343168,0.305369,0.091072,479,423,REXO1;FBLL1;LSM3;HCLS1;FARSA;PLEKHB2;STIM2;ISC...,PYGL;PZP;RAB27B;RAB2B;RAB44;RAB4A;RALGPS2;RAP1...,lscc,8.343168,113.0,8,826.5,412.0
3206,Neutrophil degranulation,0.003349,5.331940,0.054545,0.225238,479,409,HNRNPD;TNFAIP8;NUP210L;KIF1A;EIF4ENIF1;UMAD1;T...,HNRNPD;TNFAIP8;NUP210L;KIF1A;EIF4ENIF1;UMAD1;T...,luad,5.331940,313.0,8,826.5,412.0
3207,Neutrophil degranulation,0.001989,2.358804,0.816527,0.744629,479,418,UGDH;MED6;DIEXF;MED17;ERMP1;KRT33A;UROD;TMEM21...,UGDH;MED6;DIEXF;MED17;ERMP1;KRT33A;UROD;TMEM21...,ovarian,2.358804,724.0,8,826.5,412.0


In [18]:
find_neutrophil_degranulation(STEP5_ratio_of_classes)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank


In [19]:
find_neutrophil_degranulation(STEP5_signal_to_noise)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank


In [20]:
find_neutrophil_degranulation(STEP5_t_test)

,Term,es,nes,pval,fdr,geneset_size,matched_size,genes,ledge_genes,cancer_type,abs_nes,cancer_rank_abs_nes,pathway_times_enriched,pathway_avg_rank,pathway_overall_rank
4696,Neutrophil degranulation,0.005522,2.955199,0.136962,0.334243,479,427,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,PYGL;TYROBP;FCER1G;ITGAX;RAP2B;CYBB;LPCAT1;ALD...,ccrcc,2.955199,456.0,8,885.5,599.0
4697,Neutrophil degranulation,0.002632,1.086810,0.811286,0.949159,479,405,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,S100A11;HSP90AB1;CNN2;XRCC6;S100P;ILF2;PA2G4;P...,colon,1.086810,2124.0,8,885.5,599.0
4698,Neutrophil degranulation,0.005960,2.717769,0.237179,0.508867,479,427,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,EEF2;COPB1;PSMD2;DYNLT1;DSP;JUP;PKM;PSMD1;NBEA...,endometrial,2.717769,700.0,8,885.5,599.0
4699,Neutrophil degranulation,0.006129,2.254714,0.335683,0.563805,479,432,IDH1;PGM2;EEF1A1;IQGAP2;APAF1;STK10;CPNE3;TYRO...,IDH1;PGM2;EEF1A1;IQGAP2;APAF1;STK10;CPNE3;TYRO...,gbm,2.254714,1338.0,8,885.5,599.0
4700,Neutrophil degranulation,0.005945,3.208612,0.115044,0.272058,479,448,CNN2;PLAU;LPCAT1;IGF2R;FCER1G;HSP90AB1;MNDA;IT...,CNN2;PLAU;LPCAT1;IGF2R;FCER1G;HSP90AB1;MNDA;IT...,hnscc,3.208612,392.0,8,885.5,599.0
4701,Neutrophil degranulation,-0.007284,-2.939425,0.204864,0.376328,479,423,EEF2;CKAP4;DDX3X;DSP;HSP90AB1;ILF2;PLAU;PA2G4;...,TNFRSF1B;PFKL;ADGRG3;HSPA1A;HSPA6;DEFA4;CAMP;H...,lscc,2.939425,602.0,8,885.5,599.0
4702,Neutrophil degranulation,-0.006826,-3.187648,0.122283,0.303843,479,409,COPB1;SRP14;EEF2;PSMD3;HSP90AB1;PSMD6;PDAP1;DN...,GSDMD;GM2A;HLA-B;QSOX1;RHOA;TUBB4B;TCN1;SERPIN...,luad,3.187648,384.0,8,885.5,599.0
4703,Neutrophil degranulation,-0.004695,-2.395540,0.324248,0.570197,479,418,HSP90AB1;HSP90AA1;ILF2;COPB1;EEF1A1;EEF2;PKM;X...,LAIR1;LILRB2;DBNL;ARSB;CEP290;GUSB;DYNC1H1;TMC...,ovarian,2.395540,1088.0,8,885.5,599.0


## Figure 2: Expression of proteins in the neutrophil degranulation pathway

In [ ]:
# Get the ID the top expessed pathway
neutro_pathway_ids = enrichment_data.\
    loc[enrichment_data["name"] == "Neutrophil degranulation", "pathway_id"].\
    unique()

if len(neutro_pathway_ids) != 1:
    raise ValueError("Multiple pathways found.")
    
neutro_pathway_id = neutro_pathway_ids[0]

### A: With proteins in different orders for the up and down categories

In [ ]:
def expr_heatmap(expr_data, enrich_data, pathway_id, up_or_down, max_p, x_title=True):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    x_title (bool): Whether to include an x axis title.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
           ["cancer_type", "mean_expr"]]

    # Select data for the specified cancer types
    selected_expr_data = pathway_expr[pathway_expr["cancer_type"].isin(sel_cancers["cancer_type"])]

    # Sort proteins by mean expression across selected cancer types
    prot_order = selected_expr_data.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    cancer_order = sel_cancers.\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, "cancer_type"]
    
    # Map simplified_change to legend values
    selected_expr_data = selected_expr_data.assign(
        tumor_up_down=selected_expr_data["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(selected_expr_data).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (different order in each facet)" if x_title else ""
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order.values,
            axis=alt.Axis(
                title=f"Pathway {up_or_down} in tumor"
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
    ).properties(
        width=700,
        height=175
    )
    
    return plot

In [ ]:
# Plot
alt.vconcat(
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "up", max_p=0.2, x_title=False),
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "down", max_p=0.2)
)

### B: With same sorting for both

In [ ]:
def expr_heatmap_alt_sort(expr_data, enrich_data, pathway_id, max_p):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Sort proteins by mean expression across all cancer types
    prot_order = pathway_expr.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    sel_cancers = enrich_data.\
        loc[enrich_data["pathway_id"] == neutro_pathway_id, ["cancer_type", "mean_expr"]]
        
    sel_cancers = sel_cancers.\
        assign(up_or_down=np.where(sel_cancers["mean_expr"] > 0, "Pathway up in tumor", "Pathway down in tumor")).\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, ["cancer_type", "up_or_down"]]
    
    cancer_order = sel_cancers["cancer_type"].values
    
    pathway_expr = pathway_expr.merge(
        right=sel_cancers,
        how="inner",
        left_on="cancer_type",
        right_on="cancer_type",
        validate="many_to_one"
    )
    
    # Map simplified_change to legend values
    pathway_expr = pathway_expr.assign(
        tumor_up_down=pathway_expr["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(pathway_expr).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (same order in both facets)"
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order,
            axis=alt.Axis(
                title=""
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
        row=alt.Row(
            "up_or_down:N",
            sort="descending",
            title="",
            header=alt.Header(
                labelFontStyle="bold"
            )
        )
    ).properties(
        width=600,
        height=150
    ).resolve_scale(
        y="independent" # This makes it so each facet (up or down) only has the cancers that are in that category
    )
    
    return plot

In [ ]:
# Plot
expr_heatmap_alt_sort(all_expression_data, enrichment_data, neutro_pathway_id, max_p=0.2)

## Figure 3: Pathway overlay for neutrophil degranulation

In [ ]:
def pathway_overlay_wrapper(pathway_id, exp_data, enrich_data, up_or_down):
    
    prots = ut.search_reactome_proteins_in_pathways(pathway_id)
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
            "cancer_type"]
    
    # Select the desired expression data and average the simplified change across cancer types
    sel_exp = exp_data.\
        loc[
            exp_data["protein_str"].isin(prots["member"]) & 
            exp_data["cancer_type"].isin(sel_cancers),
            ["protein_str", "simplified_change"]
        ].\
        groupby("protein_str").\
        agg(np.mean).\
        rename(columns={"simplified_change": f"{up_or_down}_simp_change"})
    
    img_name = f"{TIME_START}_{pathway_id}_{up_or_down}.png"
    img_path = os.path.join(STEP7_DIR, img_name)
    
    token, _ = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=sel_exp, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    _, img_path = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
        export_path=img_path,
        image_format="png",
        display_col_idx=None,
        diagram_colors="Modern",
        overlay_colors="Standard",
        quality=10
    )

    _, url = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
    )

    return img_path, url

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

up_image_path, up_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "up")

In [ ]:
IPython.display.HTML(f'<a href={up_url}>{up_url}</a>')

In [ ]:
IPython.display.Image(up_image_path)

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

down_image_path, down_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "down")

In [ ]:
IPython.display.HTML(f'<a href={down_url}>{down_url}</a>')

In [ ]:
IPython.display.Image(down_image_path)